# MOEX Market Overview — Scraping & API (GP2)

**Фокус:** сбор данных (MOEX ISS API + web scraping), очистка, анализ динамики индексов и риск/доходность.  

In [ ]:
import re
import json
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bs4 import BeautifulSoup
from datetime import datetime

plt.rcParams["figure.figsize"] = (10, 5)

## 1) Сбор данных: MOEX ISS API

Использую функцию, которая повторяет логику из учебного проекта: забираем свечи (`candles`) постранично через параметр `start`.  
MOEX может отдавать **JSON** или **JSONP** — поэтому функция аккуратно обрабатывает оба варианта.


In [ ]:
def moex_data_history(url: str) -> pd.DataFrame:
    headers = {"User-Agent": "Mozilla/5.0"}
    all_data = []
    start = 0

    while True:
        if "start=" in url:
            full_url = re.sub(r"start=\d+", f"start={start}", url)
        else:
            full_url = f"{url}&start={start}"

        r = requests.get(full_url, headers=headers, timeout=30)
        r.raise_for_status()

        text = r.text.strip()
        # JSONP -> JSON
        if text.startswith("JSON_CALLBACK("):
            text = text.replace("JSON_CALLBACK(", "").rstrip(")")
        j = json.loads(text)

        # В режиме iss.json=extended candles лежат в j[1]['candles']
        candles = None
        try:
            candles = j[1]["candles"]
        except Exception:
            if isinstance(j, dict) and "candles" in j:
                candles = j["candles"]

        if not candles:
            break

        all_data += candles
        start += len(candles)

    return pd.DataFrame(all_data)

## 2) Индексы для сравнения

Ниже — набор индексов, которые использовались в учебном проекте.  
Даты в ссылках можно менять (`from=...`, `till=...`) под нужный период.


In [ ]:
URLS = {
  "RGBITR": "https://iss.moex.com/iss/engines/stock/markets/index/boards/RGBI/securities/RGBITR/candles.jsonp?interval=24&iss.reverse=true&from=2015-11-07&till=2025-11-10&limit=10000&sort_order=TRADEDATE&sort_order_desc=desc&iss.meta=off&iss.json=extended&callback=JSON_CALLBACK&lang=ru&start=0",
  "MCXSM": "https://iss.moex.com/iss/engines/stock/markets/index/boards/MCXSM/securities/MCXSM/candles.jsonp?interval=24&iss.reverse=true&from=2015-01-01&till=2025-11-10&limit=10000&sort_order=TRADEDATE&sort_order_desc=desc&iss.meta=off&iss.json=extended&callback=JSON_CALLBACK&lang=ru&start=0",
  "MOEXRE": "https://iss.moex.com/iss/engines/stock/markets/index/securities/MOEXRE/candles.jsonp?interval=24&iss.reverse=true&from=2020-03-20&till=2025-11-10&limit=10000&sort_order=TRADEDATE&sort_order_desc=desc&iss.meta=off&iss.json=extended&callback=JSON_CALLBACK&lang=ru",
  "RUEYBCSTR": "https://iss.moex.com/iss/engines/stock/markets/index/securities/RUEYBCSTR/candles.jsonp?interval=24&iss.reverse=true&from=2019-12-30&till=2025-11-10&limit=10000&sort_order=TRADEDATE&sort_order_desc=desc&iss.meta=off&iss.json=extended&callback=JSON_CALLBACK&lang=ru",
  "RGBI": "https://iss.moex.com/iss/engines/stock/markets/index/securities/RGBI/candles.jsonp?interval=24&iss.reverse=true&from=2015-11-01&till=2025-11-11&limit=10000&sort_order=TRADEDATE&sort_order_desc=desc&iss.meta=off&iss.json=extended&callback=JSON_CALLBACK&lang=ru",
  "MRSV": "https://iss.moex.com/iss/engines/stock/markets/index/securities/MRSV/candles.jsonp?interval=24&iss.reverse=true&from=2015-11-01&till=2025-11-10&limit=10000&sort_order=TRADEDATE&sort_order_desc=desc&iss.meta=off&iss.json=extended&callback=JSON_CALLBACK&lang=ru",
  "MRBCTR": "https://iss.moex.com/iss/engines/stock/markets/index/securities/MRBCTR/candles.jsonp?interval=24&iss.reverse=true&from=2017-12-29&till=2025-11-10&limit=10000&sort_order=TRADEDATE&sort_order_desc=desc&iss.meta=off&iss.json=extended&callback=JSON_CALLBACK&lang=ru",
  "IMOEX": "https://iss.moex.com/iss/engines/stock/markets/index/securities/IMOEX/candles.jsonp?interval=24&iss.reverse=true&from=2015-11-01&till=2025-11-11&limit=10000&sort_order=TRADEDATE&sort_order_desc=desc&iss.meta=off&iss.json=extended&callback=JSON_CALLBACK&lang=ru"
}
list(URLS.keys())

## 3) Загрузка котировок и сбор в единый датасет

На выходе получаем `final_df` с колонками по свечам (`open`, `high`, `low`, `close`, `volume`, `value`) и датой `end`.


In [ ]:
frames = []
for ticker, url in URLS.items():
    try:
        df_t = moex_data_history(url)
        df_t["Ticker"] = ticker
        frames.append(df_t)
        print(f"{ticker}: {df_t.shape}")
    except Exception as e:
        print(f"[WARN] {ticker}: не удалось загрузить ({e})")

final_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
final_df.head()

## 4) Дополнение: золото через web-scraping (как в проекте)

В учебном проекте золото подтягивалось со страницы MFD (таблица с ценой золота в руб./грамм).  

In [ ]:
def scrape_gold_mfd() -> pd.DataFrame:
    url = "https://mfd.ru/centrobank/preciousmetals/?left=0&right=-1&from=15.09.2000&till="
    resp = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    table = soup.find("table")
    if table is None:
        raise ValueError("Не нашёл таблицу с ценами золота на странице MFD")

    # Собираем строки таблицы
    rows = []
    for tr in table.find_all("tr"):
        tds = [td.get_text(strip=True) for td in tr.find_all(["td", "th"])]
        if tds:
            rows.append(tds)

    df = pd.DataFrame(rows[1:], columns=rows[0])
    return df

# Попробуем подтянуть золото (если сайт доступен)
try:
    df_gold = scrape_gold_mfd()
    df_gold.head()
except Exception as e:
    print("[WARN] Не удалось загрузить золото с MFD:", e)
    df_gold = pd.DataFrame()

## 5) Очистка и унификация форматов

- приводим числа к numeric  
- дату `end` к datetime  
- строим недельные цены и недельные доходности


In [ ]:
if final_df.empty:
    raise ValueError("final_df пустой: проверь доступность MOEX и ссылки URLS")

num_cols = ["open", "close", "high", "low", "value", "volume"]
for col in num_cols:
    if col in final_df.columns:
        final_df[col] = pd.to_numeric(final_df[col], errors="coerce")

final_df["end"] = pd.to_datetime(final_df["end"], errors="coerce")
final_df = final_df.dropna(subset=["end", "close"]).sort_values(["Ticker", "end"])

final_df.head()

In [ ]:
df_sorted = final_df.set_index("end").sort_index()

# недельная цена закрытия (последнее значение недели)
weekly_prices = (
    df_sorted
    .groupby("Ticker")["close"]
    .resample("W-FRI")
    .last()
    .reset_index()
    .rename(columns={"close": "close_w"})
)

# недельная лог-доходность
weekly_prices["ret_w"] = weekly_prices.groupby("Ticker")["close_w"].apply(lambda s: np.log(s / s.shift(1)))
weekly_prices["ret_w_pct"] = weekly_prices["ret_w"] * 100

weekly_prices.head()

## 6) Метрики риск/доходность (как в учебном проекте)

Использую те же функции:
- годовая доходность (геометрическая и простая)
- годовая волатильность
- Sharpe, Max Drawdown, Calmar
- skew/kurtosis
- CAGR


In [ ]:
def annual_returns_from_daily(ret_w):
    mean_w = ret_w.mean()
    geom_w = np.expm1(np.log1p(ret_w.dropna()).mean())
    ret_geom = (1 + geom_w)**52 - 1
    ret_simple = mean_w * 52
    return ret_geom, ret_simple

def annual_vol_from_daily(ret_w):
    std_w = ret_w.std(ddof=1)
    return std_w * np.sqrt(52)

def sharpe_ratio(ret_annual_geom, vol_annual, r_f=0.09):
    return (ret_annual_geom - r_f) / vol_annual

def max_drawdown(ret_w: pd.Series):
    wealth = (1 + ret_w.fillna(0)).cumprod()
    peak = wealth.cummax()
    drawdown = wealth / peak - 1
    return drawdown.min()

def calmar_ratio(ret_annual_geom, max_drawdown):
    return ret_annual_geom / abs(max_drawdown)

from scipy.stats import skew as skew_fn, kurtosis as kurtosis_fn
def distribution_stats(ret_w):
    r = ret_w.dropna()
    return skew_fn(r), kurtosis_fn(r, fisher=True)


def CAGR(ret_w):
    n_years = len(ret_w) / 52
    total_growth = (1 + ret_w.fillna(0)).prod()
    return total_growth**(1/n_years) - 1

In [ ]:
RISK_FREE = 0.13  # как в проекте (пример безрисковой ставки)

rows = []
for t, g in weekly_prices.groupby("Ticker"):
    ret_geom, ret_simple = annual_returns_from_daily(g["ret_w"])
    vol = annual_vol_from_daily(g["ret_w"])
    sh = sharpe_ratio(ret_geom, vol, r_f=RISK_FREE) if vol and not np.isnan(vol) else np.nan
    mdd = max_drawdown(g["ret_w"])
    cal = calmar_ratio(ret_geom, mdd) if mdd != 0 else np.nan
    sk, ku = distribution_stats(g["ret_w"])
    cagr = CAGR(g["ret_w"])

    rows.append({
        "Ticker": t,
        "Return_geom": ret_geom,
        "Return_simple": ret_simple,
        "Volatility": vol,
        "Sharpe": sh,
        "MaxDrawdown": mdd,
        "Calmar": cal,
        "Skew": sk,
        "Kurtosis": ku,
        "CAGR": cagr
    })

summary = pd.DataFrame(rows).sort_values("Sharpe", ascending=False)
summary

## 7) Визуализация: нормированная динамика цен

Сравниваем индексы “на одном графике”, нормируя цену к 1 в начале периода.


In [ ]:
df_norm = final_df.sort_values(["Ticker", "end"]).copy()
df_norm["close_norm"] = df_norm.groupby("Ticker")["close"].transform(lambda x: x / x.iloc[0])

for t in df_norm["Ticker"].unique():
    df_t = df_norm[df_norm["Ticker"] == t]
    plt.plot(df_t["end"], df_t["close_norm"], label=t)

plt.title("Нормированная динамика (close / close0)")
plt.xlabel("Дата")
plt.ylabel("Норм. цена")
plt.legend()
plt.show()

## 8) Корреляции недельных доходностей

Матрица корреляции помогает понять диверсификацию: какие индексы “ходят вместе”, а какие могут снижать общий риск портфеля.


In [ ]:
import seaborn as sns

pivot = weekly_prices.pivot(index="end", columns="Ticker", values="ret_w").dropna(how="all")
corr = pivot.corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Корреляция недельных доходностей")
plt.show()

## 9) Короткие рекомендации (как продуктовая/инвест-аналитика)

- Для **консервативного профиля** чаще подходят облигационные индексы: ниже волатильность и просадка.  
- Для **роста** — акционные индексы, но важно контролировать риск (волатильность/просадка).  
- Для **диверсификации** полезно добавлять инструменты с низкой корреляцией к акциям (например, золото — если данные доступны).  

В реальном продукте такой анализ можно упаковать в “витрину” с выбором риск-профиля и подсказками по портфельным долям.